# Create Bronze Tables for Incremental Data 

In [0]:
from pyspark.sql.functions import current_timestamp
import os

In [0]:
#do the same for openf1api

# list all openf1 api files
files=dbutils.fs.ls("/Volumes/f1_warehouse/ingestion/raw_files/openf1/")

for file in files:

    # remove the ext to get the table name
    table_name=os.path.splitext(file.name)[0]

    #read csv
    df=(spark.read.option("header", True).option("inferSchema", True).csv(file.path))

    # add metadata
    bronze_df=(df.withColumn("ingestion_timestamp", current_timestamp())
                    .withColumn("file_name", df["_metadata.file_path"]))

    # Write as a Delta table
    (
        bronze_df.write# save as df
        .format("delta")# change to delta format
        .mode("overwrite")#temp only change to append when creating ingestion pipeline
        .option("overwriteSchema","true") #overweriteschema since im including session type 
        .saveAsTable(f"f1_warehouse.bronze.openf1_{table_name}")
    )

    print(f"Created bronze.openf1_{table_name}")
